In [ ]:
from lavis.models import load_model, model_zoo

print(model_zoo)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Architectures                  Types
albef_classification           ve
albef_feature_extractor        base
albef_nlvr                     nlvr
albef_pretrain                 base
albef_retrieval                coco, flickr
albef_vqa                      vqav2
alpro_qa                       msrvtt, msvd
alpro_retrieval                msrvtt, didemo
blip_caption                   base_coco, large_coco
blip_classification            base
blip_feature_extractor         base
blip_image_text_matching       base, large
blip_nlvr                      nlvr
blip_pretrain                  base
blip_retrieval                 coco, flickr
blip_vqa                       vqav2, okvqa, aokvqa
blip2_opt                      pretrain_opt2.7b, pretrain_opt6.7b, caption_coco_opt2.7b, caption_coco_opt6.7b
blip2_t5                       pret

In [ ]:
# model = load_model('blip2', model_type='pretrain', is_eval=True, device='cpu', checkpoint='./lavis/output/BLIP2/Pretrain_stage1/20240501140/checkpoint_7.pth')

In [ ]:
from PIL import Image

example_image = Image.open("./experimental_notebooks/data/flickr_sample/878758390.jpg").convert("RGB")

In [ ]:
from torchvision import transforms

transform =  transforms.Compose([
    transforms.Resize((224, 224)),  # Resize the image
    transforms.ToTensor(),  # Convert the image to a PyTorch tensor
])

example_image_torch = transform(example_image)
print(example_image_torch.shape)
example_image_torch = example_image_torch[None, :]
print(example_image_torch.shape)



torch.Size([3, 224, 224])
torch.Size([1, 3, 224, 224])


In [ ]:
import torch
try_out_image = {'image':example_image_torch}

In [7]:
from lavis.common.registry import registry
from omegaconf import OmegaConf
from lavis.models import load_preprocess

device = torch.device("cuda") if torch.cuda.is_available() else "cpu"

model_cls = registry.get_model_class("blip2")
# print(model_cls)
model = model_cls(img_size=224,vit_precision="fp32",freeze_vit=True)

model.load_checkpoint("./lavis/output/BLIP2/Pretrain_stage1/20240501140/checkpoint_7.pth")
model.eval()
model = model.to(device)
cfg = OmegaConf.load(model_cls.default_config_path("pretrain"))
preprocess_cfg = cfg.preprocess
vis_processors, txt_processors = load_preprocess()

raw_image = example_image.resize((224,224))
image = vis_processors["eval"](raw_image).unsqueeze(0).to(device)

model.generate({"image": image})

RuntimeError: expected scalar type Float but found Half